# CineMatch: Exploratory Data Analysis

This notebook explores real MovieLens Small records. It reports the loaded dataset's values; it does not create or infer missing ratings. First download the data with `python download_data.py` from the project root.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = Path('../data/ml-latest-small')
movies = pd.read_csv(DATA / 'movies.csv')
ratings = pd.read_csv(DATA / 'ratings.csv')
tags = pd.read_csv(DATA / 'tags.csv')
sns.set_theme(style='darkgrid', palette='mako')
print(f'{len(movies):,} movies | {len(ratings):,} ratings | {len(tags):,} tags')
display(movies.head())
display(ratings.head())
display(tags.head())

## What do the columns mean?
`movies.csv` has a movie ID, title and pipe-separated genres. `ratings.csv` records an anonymous user ID, movie ID, 0.5–5 score and Unix timestamp. `tags.csv` records user-contributed text tags with user, movie and timestamp. These are observed dataset values, not generated plot descriptions.

In [ ]:
# Most-rated titles: count observed rating rows by movie.
counts = ratings.groupby('movieId').size().rename('rating_count')
rating_summary = ratings.groupby('movieId')['rating'].agg(['mean', 'count'])
title_ratings = movies.merge(rating_summary, left_on='movieId', right_index=True)
top10 = title_ratings.nlargest(10, 'count').sort_values('count')
ax = top10.plot.barh(x='title', y='count', figsize=(10, 6), legend=False, color='#9b75ff')
ax.set(title='10 most-rated movies', xlabel='Number of observed ratings', ylabel='')
plt.tight_layout()
plt.show()

**Reading this chart:** rating count measures how many users rated each title. It is not the same thing as average quality or a recommendation score.

In [ ]:
# Highest average rating, with a minimum vote count to reduce tiny-sample noise.
minimum_votes = 100
best = title_ratings[title_ratings['count'] >= minimum_votes].nlargest(10, 'mean').sort_values('mean')
ax = best.plot.barh(x='title', y='mean', figsize=(10, 6), legend=False, xlim=(0, 5), color='#53d6b0')
ax.set(title=f'Highest average rating (at least {minimum_votes} ratings)', xlabel='Mean observed rating (0.5–5)', ylabel='')
plt.tight_layout()
plt.show()

The minimum count guards against ranking a film from only a few votes. This is still a descriptive view, not an estimate of universal quality.

In [ ]:
# Ratings distribution and average ratings per user.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=ratings, x='rating', ax=axes[0], color='#9b75ff')
axes[0].set(title='Distribution of rating values', xlabel='Rating', ylabel='Count')
per_user = ratings.groupby('userId')['rating'].agg(['mean', 'count'])
sns.histplot(per_user['mean'], bins=20, ax=axes[1], color='#53d6b0')
axes[1].set(title='Each user’s average rating', xlabel='Mean rating per user', ylabel='Number of users')
plt.tight_layout()
plt.show()

**Reading these charts:** the left chart shows which score values users chose most often. The right chart treats each user once and reveals whether users tend to use the scale similarly or have different average scoring habits.

In [ ]:
# Genre counts: a movie may belong to several genres, so counts overlap.
genre_counts = (movies.assign(genre=movies['genres'].fillna('').str.split('|'))
                .explode('genre')
                .query("genre != '' and genre != '(no genres listed)'")
                .groupby('genre').size().sort_values(ascending=False))
ax = genre_counts.sort_values().plot.barh(figsize=(9, 7), color='#53d6b0')
ax.set(title='Movies per genre', xlabel='Number of movies with this genre', ylabel='Genre')
plt.tight_layout()
plt.show()

# User activity pattern: number of ratings provided by each user.
activity = ratings.groupby('userId').size()
print(activity.describe())
sns.histplot(activity, bins=30, color='#9b75ff')
plt.title('Number of ratings per user')
plt.xlabel('Ratings contributed by one user')
plt.tight_layout()
plt.show()

**Reading these charts:** genre memberships overlap, so the genre totals add up to more than the number of movies. The user activity histogram describes how many ratings each anonymous user contributed. It helps explain why collaborative methods may have uneven amounts of evidence per user.

## Next steps
The content-based system uses title, genre and tags; popularity uses average ratings with rating counts; the optional collaborative baseline uses user–movie ratings. For evaluation, make a per-user time-aware holdout, generate recommendations without seeing held-out ratings, and report Precision@K / Recall@K. A rating predictor should report RMSE on held-out ratings. Do not interpret exploratory plots as proof of recommendation accuracy.